In [ ]:
import sys
from pathlib import Path

# Walk up from the notebook's location until we find the directory
# containing the `eccopy` package folder, then add it to sys.path.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "eccopy").is_dir():
        sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Could not find the eccopy package directory above " + str(here))


# EccoPy-3D Workflow Template

Minimal template: load a real LROSE-ECCO case, clean it, run EccoPy, load
LROSE's real output. Comparison / metrics / plotting are left to you.

**Case folder** should contain `ECCO_IN_*.npz` (DBZ, X_km, Y_km, Z_km,
optionally height/temp) and `ECCO_OUT_*.npz` (ECHOTYPE, CONVECTIVITY,
TEXTURE) — plus the original `Ecco*.spol`/`.iop`/`.sea` param file(s).

EccoPy-3D has been validated against real LROSE-ECCO truth on 3
independent cases (SPOL, WRF, SEA — see README "Validation status"),
achieving 99.5%+ both-echo agreement on all of them. This notebook is
the same load -> clean -> run -> load-reference scaffold used to do
that; swap `CASE_DIR` / `PARAM_FILE` below to point at a different
case, or a new one.

Things worth knowing before you start (see README "Validation status"
for full details):
1. Real MDV-derived data commonly uses a literal `-9999.0` fill value,
   not NaN — clean it yourself (Step 2) or EccoPy will treat it as real
   (very low) reflectivity. Some cases use OTHER fill sentinels too
   (e.g. the original WRF case's DBZ field used `-40.0` for ~37% of
   the volume) — always check `np.unique()` / a histogram of your raw
   DBZ array for suspiciously common round values before trusting it.
   `min_valid_dbz` filtering (Step 3) will null anything below
   threshold either way, but it's worth knowing what's in your data.
2. If the param file has `vert_levels_type = VERT_LEVELS_BY_HT`, the
   shallow/deep boundaries are pure height constants — build `height` by
   simply broadcasting `Z_km` across (Y, X); no separate temperature
   field is needed even for full sub-classification.
3. Pull EVERY parameter below directly from the case's `Ecco.*` param
   file rather than trusting the package defaults — several real
   validated cases use non-default values (e.g. SEA's
   `min_valid_dbz=-10`, `texture_limit_high=23`,
   `shallow_threshold_ht=3.0`, `deep_threshold_ht=5.0`).

In [ ]:
import numpy as np
import glob
import warnings

from eccopy import eccopy3d
from eccopy.params import WindowSpec, TextureParams, ClassificationParams, VerticalParams


## Step 1 — Locate the case files

In [ ]:
CASE_DIR = "ECCO_3D_CASE_2_WRF"   # <-- change to your case folder

in_files = glob.glob(f"{CASE_DIR}/**/*IN*.npz", recursive=True)
out_files = sorted(glob.glob(f"{CASE_DIR}/**/*OUT*.npz", recursive=True))
param_files = [f for f in glob.glob(f"{CASE_DIR}/**/*", recursive=True)
               if f.split('.')[-1] in ('spol', 'iop', 'sea')]

print("Input file(s):", in_files)
print("Output file(s):", out_files)
print("Param file(s):", param_files)


In [ ]:
INPUT_FILE = in_files[0]
OUTPUT_FILE = out_files[0]   # <-- change index if there are multiple variants

data_in = np.load(INPUT_FILE)
data_out = np.load(OUTPUT_FILE)

print("Input fields:", data_in.files)
print("Output fields:", data_out.files)


## Step 2 — Load and clean the data

Clean literal `-9999.0` fill values to NaN before calling EccoPy.

In [ ]:
dbz = data_in['DBZ'].astype(float)
dbz[dbz == -9999.0] = np.nan   # <-- clean fill values; repeat for other arrays if needed

x_km = data_in['X_km'].astype(float)
y_km = data_in['Y_km'].astype(float)
z_km = data_in['Z_km'].astype(float)

nz, ny, nx = dbz.shape

# For VERT_LEVELS_BY_HT param files (the common case): height is just
# Z_km broadcast across the horizontal plane, no separate field needed.
height_km = np.broadcast_to(z_km[:, None, None], dbz.shape).copy()

# If your case genuinely needs a temperature field instead (VERT_LEVELS_BY_TEMP),
# load and clean it here:
# temp_c = data_in['TEMP'].astype(float)
# temp_c[temp_c == -9999.0] = np.nan


## Step 3 — Set parameters

Pull these directly from the case's `Ecco.*` param file. The values below
match `ECCO_3D_CASE_2_WRF/Ecco.iop` — update for your own case.

In [ ]:
TEXTURE_RADIUS_KM = 4.0
SHALLOW_THRESHOLD_HT = 4.5
DEEP_THRESHOLD_HT = 9.0
MIN_VALID_VOLUME_FOR_CONVECTIVE = 30.0   # check this per-case; the package
                                           # default (20.0) is NOT always
                                           # what the real run used

window = WindowSpec((TEXTURE_RADIUS_KM, 'km'))
texture_params = TextureParams(
    dbz_base=0.0, min_valid_dbz=0.0, min_frac_texture=0.25, min_frac_fit=0.67,
    texture_limit_low=0.0, texture_limit_high=30.0,
)
class_params = ClassificationParams(
    min_convectivity_for_convective=0.5, max_convectivity_for_stratiform=0.4,
    use_dual_thresholds=True, secondary_convectivity=0.65,
    all_subclumps_min_area_frac=0.33, each_subclump_min_area_frac=0.02,
    min_valid_volume_for_convective=MIN_VALID_VOLUME_FOR_CONVECTIVE,
    min_vert_extent_for_convective=1.0,
    min_conv_fraction_for_deep=0.05, min_conv_fraction_for_shallow=0.95,
    max_shallow_conv_fraction_for_elevated=0.05, max_deep_conv_fraction_for_elevated=0.25,
    min_strat_fraction_for_strat_below=0.9,
)
vert_params = VerticalParams(
    shallow_threshold_ht=SHALLOW_THRESHOLD_HT,
    deep_threshold_ht=DEEP_THRESHOLD_HT,
)


## Step 4 — Run EccoPy

This can take a little while on large grids (a 750x750x46 grid takes
roughly 15-30s once Numba is warmed up). If you have `topo`/`terrain_ht`
for a case with real terrain, add those too — see README.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    result = eccopy3d.run(
        dbz, coords_z=z_km, coords_y=y_km, coords_x=x_km,
        height=height_km,
        window=window,
        texture_params=texture_params,
        class_params=class_params,
        vert_params=vert_params,
        # topo=topo_km, terrain_ht=terrain_ht_km,
    )

our_echo = result.echo_type
our_conv = result.convectivity
our_texture = result.texture

print("echo_type shape:", our_echo.shape)
print("n_clumps:", result.n_clumps)
print("codes present:", sorted(set(our_echo[our_echo > 0].tolist())))


## Step 5 — LROSE's real output

Loaded and ready for comparison.

In [ ]:
lrose_echo = data_out['ECHOTYPE']
lrose_conv = data_out['CONVECTIVITY'] if 'CONVECTIVITY' in data_out.files else None
lrose_texture = data_out['TEXTURE'] if 'TEXTURE' in data_out.files else None


## Your analysis

`our_echo`, `our_conv`, `our_texture` vs. `lrose_echo`, `lrose_conv`,
`lrose_texture` — ready to compare, score, or plot however's useful.